# POMP v2 Diagnostics — Overnight Run

Configured for an overnight run (~6 hours) to get clean multi-start convergence.

**Key settings:**
- `N_STARTS = 15`
- `MAX_ITER_PER_START = 1000` (5× previous)
- `N_PARTICLES = 1500` (3× previous)
- Smaller perturbation scales — starts closer to theta_hat
- Soft penalty on non-stationary VAR coefficients
- Incremental pickle save after every start (crash-safe)

**Before running:**
- Windows: Settings → Power → Sleep = Never
- Keep plugged in, not on battery
- Close heavy apps
- Don't close browser window (can minimize)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import t as student_t, norm
import matplotlib.pyplot as plt
import pickle
import time

np.set_printoptions(suppress=True, precision=4)
plt.rcParams["figure.dpi"] = 100

## 0. Load data + v2 model functions

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO"
price = pd.read_csv(url)
price["observation_date"] = pd.to_datetime(price["observation_date"], errors="coerce")
price["DCOILWTICO"] = pd.to_numeric(price["DCOILWTICO"], errors="coerce")
price = (
    price.rename(columns={"observation_date": "date", "DCOILWTICO": "price"})
    .dropna().query("date >= '2014-01-01'").query("price > 0")
    .sort_values("date").reset_index(drop=True)
)
price["log_price"] = np.log(price["price"])
price["return"] = 100 * price["log_price"].diff()
df = price.dropna().copy().reset_index(drop=True)
y_np = df["return"].to_numpy()
dates = df["date"].to_numpy()
T = len(y_np)
print(f"T = {T}")

In [ ]:
PARAM_NAMES = ["alpha1", "beta1", "alpha2", "beta2",
               "log_sigma1", "log_sigma2",
               "log_s1", "log_s2", "log_s3",
               "mu", "gamma", "log_nu_minus2"]
N_PARAMS = 12

def unpack(theta):
    (a1, b1, a2, b2, ls1, ls2, lS1, lS2, lS3, mu, gamma, lnu) = theta
    return dict(alpha1=a1, beta1=b1, alpha2=a2, beta2=b2,
                sigma1=np.exp(ls1), sigma2=np.exp(ls2),
                s1=np.exp(lS1), s2=np.exp(lS2), s3=np.exp(lS3),
                mu=mu, gamma=gamma, nu=2.0 + np.exp(lnu))

def softmax_3class(nu1, nu2):
    m = np.maximum(np.maximum(nu1, nu2), 0.0)
    e1, e2, e3 = np.exp(nu1 - m), np.exp(nu2 - m), np.exp(-m)
    s = e1 + e2 + e3
    return e1/s, e2/s, e3/s

def particle_filter_loglik(theta, y, N=1000, seed=42, return_filtered=False):
    """v2 particle filter with clip safeguard."""
    p = unpack(theta)
    a1, b1, a2, b2 = p["alpha1"], p["beta1"], p["alpha2"], p["beta2"]
    sn1, sn2 = p["sigma1"], p["sigma2"]
    s1_, s2_, s3_ = p["s1"], p["s2"], p["s3"]
    mu, gamma, nu = p["mu"], p["gamma"], p["nu"]

    rng = np.random.default_rng(seed)
    u1 = 0.1 * rng.standard_normal(N)
    u2 = 0.1 * rng.standard_normal(N)

    T = len(y)
    loglik = 0.0
    r_prev = 0.0
    log_norm_const = (gammaln((nu+1)/2) - gammaln(nu/2) - 0.5*np.log(nu*np.pi))
    nu_half_plus = (nu + 1) / 2

    if return_filtered:
        filt = np.zeros((T, 3))

    for t in range(T):
        u1_new = np.clip(a1*u1 + b1*u2 + sn1*rng.standard_normal(N), -50, 50)
        u2_new = np.clip(a2*u1 + b2*u2 + sn2*rng.standard_normal(N), -50, 50)

        x1, x2, x3 = softmax_3class(u1_new, u2_new)
        sigma_t = s1_*x1 + s2_*x2 + s3_*x3
        mean_obs = mu + gamma*r_prev
        z = (y[t] - mean_obs) / sigma_t
        log_w = log_norm_const - np.log(sigma_t) - nu_half_plus * np.log1p(z*z/nu)

        max_lw = np.max(log_w)
        w = np.exp(log_w - max_lw)
        sum_w = np.sum(w)
        loglik += max_lw + np.log(sum_w) - np.log(N)

        w_norm = w / sum_w
        if return_filtered:
            filt[t, 0] = np.sum(w_norm * x1)
            filt[t, 1] = np.sum(w_norm * x2)
            filt[t, 2] = np.sum(w_norm * x3)

        u0 = rng.uniform() / N
        positions = u0 + np.arange(N) / N
        cumw = np.cumsum(w_norm)
        idx = np.clip(np.searchsorted(cumw, positions), 0, N - 1)
        u1, u2 = u1_new[idx], u2_new[idx]
        r_prev = y[t]

    if return_filtered:
        return loglik, filt
    return loglik

In [ ]:
try:
    with open("pomp_results_v2.pkl", "rb") as f:
        saved = pickle.load(f)
    theta_hat = saved["theta_hat"]
    print("Loaded theta_hat from pomp_results_v2.pkl")
except FileNotFoundError:
    theta_hat = np.array([
        0.9930, -0.0000, -0.0001, 0.9921,
        np.log(0.5235), np.log(0.4970),
        np.log(1.1930), np.log(2.5930), np.log(8.3086),
        0.0002, -0.0519,
        np.log(6.1718 - 2.0),
    ])
    print("Using hard-coded theta_hat")

p_hat = unpack(theta_hat)
for name, val in p_hat.items():
    print(f"  {name:>10s} = {val: .4f}")

## Optional: Pre-run speed check

Run this to estimate total time before committing to the overnight run.
Total estimate ≈ single_pf_time × N_STARTS × MAX_ITER / 60 minutes.

In [ ]:
t0 = time.time()
ll = particle_filter_loglik(theta_hat, y_np, N=1500, seed=42)
single_pf_time = time.time() - t0
print(f"Single PF run (N=1500): {single_pf_time:.2f}s, loglik = {ll:.2f}")

# Estimate for the main run
N_STARTS_PLANNED = 15
MAX_ITER_PLANNED = 1000
est_hours = single_pf_time * N_STARTS_PLANNED * MAX_ITER_PLANNED / 3600
print(f"\nEstimated total time for {N_STARTS_PLANNED} starts × {MAX_ITER_PLANNED} iter: {est_hours:.1f} hours")
print("If this is > your available time, reduce N_STARTS or MAX_ITER in the next cell.")

## 1. Multi-start search — overnight configuration

**Expected runtime: ~6 hours.** Saves progress incrementally to `multistart_progress.pkl`.

In [ ]:
N_STARTS = 15
MAX_ITER_PER_START = 1000
N_PARTICLES = 1500

# Smaller perturbations — starts closer to theta_hat for easier convergence
PERTURB_SCALES = np.array([
    0.01,  # alpha1
    0.03,  # beta1
    0.03,  # alpha2
    0.01,  # beta2
    0.2,   # log_sigma1
    0.2,   # log_sigma2
    0.2,   # log_s1
    0.2,   # log_s2
    0.2,   # log_s3
    0.03,  # mu
    0.03,  # gamma
    0.2,   # log_nu_minus2
])

def safe_perturbed_start(theta_hat, rng):
    theta0 = theta_hat + PERTURB_SCALES * rng.standard_normal(N_PARAMS)
    theta0[0] = np.clip(theta0[0], 0.85, 0.999)
    theta0[3] = np.clip(theta0[3], 0.85, 0.999)
    theta0[1] = np.clip(theta0[1], -0.1, 0.1)
    theta0[2] = np.clip(theta0[2], -0.1, 0.1)
    return theta0

trajectories = []
rng = np.random.default_rng(0)
start_time = time.time()

for s in range(N_STARTS):
    theta0 = safe_perturbed_start(theta_hat, rng)
    theta_list = [theta0.copy()]
    ll_list = [particle_filter_loglik(theta0, y_np, N=N_PARTICLES, seed=42)]

    def obj(theta):
        # Soft penalty on non-stationary AR — keeps optimizer away from bad region
        penalty = 0.0
        if abs(theta[0]) > 0.999:
            penalty += 1e5 * (abs(theta[0]) - 0.999) ** 2
        if abs(theta[3]) > 0.999:
            penalty += 1e5 * (abs(theta[3]) - 0.999) ** 2
        ll = particle_filter_loglik(theta, y_np, N=N_PARTICLES, seed=42)
        theta_list.append(theta.copy())
        ll_list.append(ll)
        return -ll + penalty

    res = minimize(obj, theta0, method="Nelder-Mead",
                   options={"maxiter": MAX_ITER_PER_START, "adaptive": True, "disp": False})

    trajectories.append({
        "theta_traj": np.array(theta_list),
        "ll_traj": np.array(ll_list),
        "final_ll": -res.fun,
    })
    elapsed = time.time() - start_time
    print(f"  start {s+1:>2d}/{N_STARTS}: initial ll={ll_list[0]:.1f} -> final ll={-res.fun:.1f}  ({elapsed/60:.1f} min)")

    # Incremental save — if it crashes overnight, we still have partial results
    with open("multistart_progress.pkl", "wb") as f:
        pickle.dump(trajectories, f)

print(f"\nAll {N_STARTS} starts done in {(time.time()-start_time)/60:.1f} min")

final_lls = np.array([t["final_ll"] for t in trajectories])
initial_lls = np.array([t["ll_traj"][0] for t in trajectories])
print(f"\nInitial ll range: [{initial_lls.min():.1f}, {initial_lls.max():.1f}]")
print(f"Final   ll range: [{final_lls.min():.1f}, {final_lls.max():.1f}]")
print(f"Best final ll   : {final_lls.max():.1f}")
print(f"Within 5 of best: {(final_lls > final_lls.max() - 5).sum()}/{N_STARTS}")

In [ ]:
# If you're resuming from a crashed run, uncomment:
# with open("multistart_progress.pkl", "rb") as f:
#     trajectories = pickle.load(f)

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

def transform_display(theta_matrix):
    T_ = theta_matrix.copy()
    T_[:, 4] = np.exp(T_[:, 4])
    T_[:, 5] = np.exp(T_[:, 5])
    T_[:, 6] = np.exp(T_[:, 6])
    T_[:, 7] = np.exp(T_[:, 7])
    T_[:, 8] = np.exp(T_[:, 8])
    T_[:, 11] = 2.0 + np.exp(T_[:, 11])
    return T_

display_names = ["alpha1", "beta1", "alpha2", "beta2",
                 "sigma1", "sigma2", "s1", "s2", "s3",
                 "mu", "gamma", "nu"]
colors = plt.cm.tab20(np.linspace(0, 1, len(trajectories)))

for p_idx in range(N_PARAMS):
    ax = axes[p_idx]
    for s_idx, traj in enumerate(trajectories):
        th_disp = transform_display(traj["theta_traj"])
        ax.plot(th_disp[:, p_idx], color=colors[s_idx], alpha=0.6, linewidth=0.8)
    ax.set_title(display_names[p_idx])
    ax.grid(alpha=0.3)

ax = axes[N_PARAMS]
final_lls_plot = np.array([t["final_ll"] for t in trajectories])
V2_REF = -6819.70   # Final theta_hat loglik at N=3000 × 10 seeds
for s_idx, traj in enumerate(trajectories):
    ax.plot(traj["ll_traj"], color=colors[s_idx], alpha=0.6, linewidth=0.8)
ax.set_title("loglik")
ax.grid(alpha=0.3)
ax.axhline(V2_REF, color="black", linestyle="--", alpha=0.5, label=f"v2 ref ({V2_REF:.1f})")
ax.legend()

# Auto-zoom to informative range
ll_lo = max(final_lls_plot.min() - 30, V2_REF - 150)
ll_hi = V2_REF + 10
ax.set_ylim(ll_lo, ll_hi)

for i in range(N_PARAMS + 1, len(axes)):
    axes[i].axis("off")

fig.suptitle("Multi-start Search Diagnostic (POMP v2)", fontsize=14)
fig.supxlabel("iteration")
fig.supylabel("value")
plt.tight_layout()
plt.savefig("diagnostic_multistart.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Filtered state probabilities over time

In [ ]:
_, filtered_probs = particle_filter_loglik(theta_hat, y_np, N=5000, seed=42, return_filtered=True)
print(f"filtered_probs shape: {filtered_probs.shape}")

vols = np.array([p_hat["s1"], p_hat["s2"], p_hat["s3"]])
vol_order = np.argsort(vols)
state_labels = ["", "", ""]
state_labels[vol_order[0]] = f"calm (σ≈{vols[vol_order[0]]:.2f})"
state_labels[vol_order[1]] = f"stress (σ≈{vols[vol_order[1]]:.2f})"
state_labels[vol_order[2]] = f"shock (σ≈{vols[vol_order[2]]:.2f})"
print(f"State labels: {state_labels}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={"height_ratios": [1, 2]})

axes[0].plot(dates, y_np, linewidth=0.4, color="steelblue")
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_ylabel("daily return (%)")
axes[0].set_title("WTI oil daily returns")
axes[0].grid(alpha=0.3)

sorted_filt = filtered_probs[:, vol_order]
axes[1].stackplot(dates,
                   sorted_filt[:, 0], sorted_filt[:, 1], sorted_filt[:, 2],
                   labels=[state_labels[vol_order[0]],
                           state_labels[vol_order[1]],
                           state_labels[vol_order[2]]],
                   colors=["#9EC7EC", "#F4B27A", "#D15E5E"], alpha=0.85)
axes[1].set_ylabel("P(regime | data)")
axes[1].set_xlabel("date")
axes[1].set_title("Filtered regime probabilities")
axes[1].legend(loc="upper left", fontsize=9)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("filtered_states.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Likelihood profile for nu

Cost: ~15 minutes.

In [ ]:
nu_grid = np.array([3.5, 4.5, 5.5, 6.5, 7.5, 9.0, 12.0])
N_GRID = len(nu_grid)
profile_lls = np.zeros(N_GRID)

start = time.time()
for i, nu_val in enumerate(nu_grid):
    fixed_log_nu = np.log(nu_val - 2.0)

    def obj(theta_rest):
        full = np.concatenate([theta_rest, [fixed_log_nu]])
        return -particle_filter_loglik(full, y_np, N=500, seed=42)

    theta_rest0 = theta_hat[:-1].copy()
    res = minimize(obj, theta_rest0, method="Nelder-Mead",
                   options={"maxiter": 150, "adaptive": True})
    profile_lls[i] = -res.fun
    print(f"  nu={nu_val:.1f}: loglik={profile_lls[i]:.2f}  ({(time.time()-start)/60:.1f} min)")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(nu_grid, profile_lls, "o-", color="steelblue")
best_ll = profile_lls.max()
ax.axhline(best_ll - 1.92, color="red", linestyle="--", label="95% CI cutoff")
ax.set_xlabel("nu (Student-t df)")
ax.set_ylabel("profile log-likelihood")
ax.set_title("Profile likelihood for nu")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("profile_nu.png", dpi=120, bbox_inches="tight")
plt.show()

ci_mask = profile_lls > best_ll - 1.92
if ci_mask.sum() >= 2:
    ci_lo, ci_hi = nu_grid[ci_mask].min(), nu_grid[ci_mask].max()
    print(f"\n95% profile CI for nu: [{ci_lo:.2f}, {ci_hi:.2f}]")

## 4. Simulated vs observed returns

In [ ]:
def simulate_from_model(theta, T_sim, seed=0):
    p = unpack(theta)
    rng = np.random.default_rng(seed)
    nu1, nu2 = 0.0, 0.0
    r_prev = 0.0
    y_sim = np.zeros(T_sim)
    for t in range(T_sim):
        nu1_new = np.clip(p["alpha1"]*nu1 + p["beta1"]*nu2 + p["sigma1"]*rng.standard_normal(), -50, 50)
        nu2_new = np.clip(p["alpha2"]*nu1 + p["beta2"]*nu2 + p["sigma2"]*rng.standard_normal(), -50, 50)
        logits = np.array([nu1_new, nu2_new, 0.0])
        logits -= logits.max()
        ex = np.exp(logits)
        x = ex / ex.sum()
        sigma_t = p["s1"]*x[0] + p["s2"]*x[1] + p["s3"]*x[2]
        mean_obs = p["mu"] + p["gamma"]*r_prev
        y_sim[t] = mean_obs + sigma_t * student_t.rvs(p["nu"], random_state=rng)
        nu1, nu2, r_prev = nu1_new, nu2_new, y_sim[t]
    return y_sim

sims = [simulate_from_model(theta_hat, T, seed=s) for s in range(4)]

fig, axes = plt.subplots(5, 1, figsize=(14, 10))
axes[0].plot(y_np, color="black", linewidth=0.4)
axes[0].set_title("Observed WTI returns")
axes[0].set_ylabel("return (%)")
axes[0].grid(alpha=0.3)
axes[0].set_ylim(-20, 20)

for i, sim in enumerate(sims):
    axes[i+1].plot(sim, color="steelblue", linewidth=0.4)
    axes[i+1].set_title(f"Simulated returns, seed={i}")
    axes[i+1].set_ylabel("return (%)")
    axes[i+1].grid(alpha=0.3)
    axes[i+1].set_ylim(-20, 20)

axes[-1].set_xlabel("trading day")
plt.tight_layout()
plt.savefig("sim_vs_observed.png", dpi=120, bbox_inches="tight")
plt.show()

print("Summary statistics comparison:")
print(f"{'stat':<10}{'observed':>12}{'sim mean':>12}{'sim sd':>12}")
def stats(x):
    return {"mean": x.mean(), "sd": x.std(),
            "skew": ((x - x.mean())**3).mean() / x.std()**3,
            "kurt": ((x - x.mean())**4).mean() / x.std()**4,
            "max|x|": np.abs(x).max()}
obs_s = stats(y_np)
sim_stats = [stats(s) for s in sims]
for key in obs_s:
    sim_vals = [s[key] for s in sim_stats]
    print(f"  {key:<8}{obs_s[key]:>12.3f}{np.mean(sim_vals):>12.3f}{np.std(sim_vals):>12.3f}")

## 5. QQ plot of standardized residuals

In [ ]:
r_lag = np.concatenate([[0.0], y_np[:-1]])
cond_mean = p_hat["mu"] + p_hat["gamma"] * r_lag
vols_vec = np.array([p_hat["s1"], p_hat["s2"], p_hat["s3"]])
cond_sd = filtered_probs @ vols_vec
z = (y_np - cond_mean) / cond_sd
print(f"Standardized residual stats: mean={z.mean():.3f}, sd={z.std():.3f}")
print(f"Theoretical sd for t_{p_hat['nu']:.2f}: {np.sqrt(p_hat['nu']/(p_hat['nu']-2)):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

theoretical_q_t = student_t.ppf(np.linspace(0.001, 0.999, len(z)), p_hat["nu"])
empirical_q = np.sort(z)
axes[0].scatter(theoretical_q_t, empirical_q, s=3, alpha=0.5)
axes[0].plot([theoretical_q_t.min(), theoretical_q_t.max()],
             [theoretical_q_t.min(), theoretical_q_t.max()], "r--", alpha=0.7)
axes[0].set_xlabel(f"theoretical quantile (t_{p_hat['nu']:.1f})")
axes[0].set_ylabel("empirical quantile")
axes[0].set_title("QQ: standardized residuals vs Student-t")
axes[0].grid(alpha=0.3)

theoretical_q_n = norm.ppf(np.linspace(0.001, 0.999, len(z)))
axes[1].scatter(theoretical_q_n, empirical_q, s=3, alpha=0.5, color="orange")
axes[1].plot([theoretical_q_n.min(), theoretical_q_n.max()],
             [theoretical_q_n.min(), theoretical_q_n.max()], "r--", alpha=0.7)
axes[1].set_xlabel("theoretical quantile (Normal)")
axes[1].set_ylabel("empirical quantile")
axes[1].set_title("QQ: vs Normal (comparison)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("qq_plot.png", dpi=120, bbox_inches="tight")
plt.show()